# Investigating the transcriptome, surface phenotype and CNV of patient specific cluster

### Leiden cluster 9

Investigate the transcriptome and surface phenotype for this patient. This is patient P18

In [ ]:
import scanpy as sc
import infercnvpy as cnv
import matplotlib.pyplot as plt
import warnings
import pandas as pd
import seaborn as sns
import muon as mu

warnings.filterwarnings('ignore')
sc.settings.verbosity = 3
plt.rcParams['figure.dpi'] = 200
plt.rcParams['figure.facecolor'] = 'white'
plt.rcParams['figure.figsize'] = (5, 5)

#### Load in the markers file

In [ ]:
markersfile = pd.read_csv("data/2025_03_12_Markers_for_Priscilla(Sheet1).csv")

In [ ]:
markersfile

#### Load in the adata

In [ ]:
adata_protein = sc.read_h5ad("/home/prisb/projects/henry_hashtag_experiment/exmds/combined_protein.h5ad")
adata_protein.obs.index = adata_protein.obs["dataset"].astype('str') + "_" + adata_protein.obs.index

In [ ]:
adata_protein

In [ ]:
adata_rna = sc.read_h5ad("12_InferCNV.h5ad")
adata_rna.obs.index = adata_rna.obs["dataset_name"].astype('str') + "_" + adata_rna.obs.index

In [ ]:
adata_rna

In [ ]:
mdata = mu.MuData({"rna":adata_rna, "protein":adata_protein})

In [ ]:
#transfer metadata from rna to protein to be accesible by both
mdata['protein'].obs['patient_alias'] = mdata['rna'].obs['patient_alias']
mdata['protein'].obs['timepoint_coarse'] = mdata['rna'].obs['timepoint_coarse']
mdata['protein'].obs['new_celltype'] = mdata['rna'].obs['new_celltype']
mdata['protein'].obs['cnv_leiden'] = mdata['rna'].obs['cnv_leiden']
mdata['protein'].obs['leiden'] = mdata['rna'].obs['leiden']

In [ ]:
mu.pp.intersect_obs(mdata)

In [ ]:
mdata

#### patient P18 

In [ ]:
p18= mdata[mdata['rna'].obs['patient_alias'] == "P18"].copy()

In [ ]:
sc.pl.dotplot(p18['protein'], markersfile[markersfile["Cluster"] == 9]["Marker"].tolist(), groupby="leiden", dendrogram=False)

In [ ]:
p18_protein = p18['protein'].copy()
p18_protein = p18_protein[~p18_protein.obs["leiden"].isin(["0","20"])].copy()
p18_protein

In [ ]:
sc.tl.rank_genes_groups(p18_protein, groupby="leiden", key_added="ADT_rank_genes_groups")

In [ ]:
sc.pl.rank_genes_groups_matrixplot(p18_protein,groups=["9","13","15"] ,groupby="leiden", 
key="ADT_rank_genes_groups", swap_axes=True)

In [ ]:
sc.pl.umap(mdata['rna'][mdata['rna'].obs["patient_alias"] == "P18"], color=["leiden"], legend_loc="on data")